# LightGCN — Graph-Based Board Game Recommender


In [29]:
# CELL 1 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
# CELL 2 - Install Dependencies
!pip install -q polars pyarrow scipy

In [31]:
# CELL 3 - Imports and Configuration
import polars as pl
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time
import gc
import copy
import itertools

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

DRIVE_PATH = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
SPLIT_DIR  = f'{DRIVE_PATH}/processed_data/final_foundation/splits'
MODEL_PATH = f'{DRIVE_PATH}/processed_data/lightgcn_best_model.pt'

K_METRICS  = 10
THRESHOLD  = 7.0

MAX_EPOCHS = 12
PATIENCE   = 2
BATCH_SIZE = 8192

EMB_DIM_OPTIONS = [32, 64]
N_LAYER_OPTIONS = [2]
LR_OPTIONS      = [1e-3, 3e-3]
WEIGHT_DECAY    = 1e-5

Using device: cuda


In [32]:
# CELL 4 - Load Shared Data
print('Loading shared train/val/test splits...')

train_df = (
    pl.scan_parquet(f'{SPLIT_DIR}/train.parquet')
      .select(['user_id', 'item_id', 'Rating'])
      .collect()
)

val_df = (
    pl.scan_parquet(f'{SPLIT_DIR}/val.parquet')
      .select(['user_id', 'item_id', 'Rating'])
      .collect()
)

test_df = (
    pl.scan_parquet(f'{SPLIT_DIR}/test.parquet')
      .select(['user_id', 'item_id', 'Rating'])
      .collect()
)

print(f'Train rows: {len(train_df):,}')
print(f'Val rows  : {len(val_df):,}')
print(f'Test rows : {len(test_df):,}')

Loading shared train/val/test splits...
Train rows: 14,929,223
Val rows  : 1,856,255
Test rows : 2,124,050


In [33]:
# CELL 5 - Build Train-Only ID Space and Fallback Statistics
print('Building train-only ID mappings and fallback statistics...')

unique_users = train_df['user_id'].unique().sort().to_list()
unique_items = train_df['item_id'].unique().sort().to_list()

user2idx = {u: i for i, u in enumerate(unique_users)}
item2idx = {it: i for i, it in enumerate(unique_items)}

N_USERS = len(user2idx)
N_ITEMS = len(item2idx)

GLOBAL_MEAN = float(train_df['Rating'].mean())

user_mean_df = train_df.group_by('user_id').agg(
    pl.col('Rating').mean().alias('user_mean')
)

item_mean_df = train_df.group_by('item_id').agg(
    pl.col('Rating').mean().alias('item_mean')
)

print(f'Unique train users: {N_USERS:,}')
print(f'Unique train items: {N_ITEMS:,}')
print(f'Global mean       : {GLOBAL_MEAN:.4f}')

Building train-only ID mappings and fallback statistics...
Unique train users: 332,020
Unique train items: 21,925
Global mean       : 7.1175


In [34]:
# CELL 6 - Mapping Helpers
def map_ids_with_flags(df, user2idx=user2idx, item2idx=item2idx):
    out = df.with_columns([
        pl.col('user_id').replace(user2idx, default=None).alias('uid'),
        pl.col('item_id').replace(item2idx, default=None).alias('iid'),
    ]).with_columns([
        pl.col('uid').is_not_null().alias('user_seen'),
        pl.col('iid').is_not_null().alias('item_seen'),
        (pl.col('uid').is_not_null() & pl.col('iid').is_not_null()).alias('both_seen')
    ])
    return out

train_mapped = map_ids_with_flags(train_df)
val_mapped   = map_ids_with_flags(val_df)
test_mapped  = map_ids_with_flags(test_df)

print(f'Train rows with both seen: {train_mapped.filter(pl.col("both_seen")).height:,}')
print(f'Val rows with both seen  : {val_mapped.filter(pl.col("both_seen")).height:,}')
print(f'Test rows with both seen : {test_mapped.filter(pl.col("both_seen")).height:,}')

/tmp/ipykernel_1054/2036193345.py:4: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col('user_id').replace(user2idx, default=None).alias('uid'),
/tmp/ipykernel_1054/2036193345.py:5: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col('item_id').replace(item2idx, default=None).alias('iid'),


Train rows with both seen: 14,929,223
Val rows with both seen  : 1,856,255
Test rows with both seen : 2,044,695


In [35]:
# CELL 7 - Build Normalized Bipartite Adjacency from Full Train
def build_norm_adj(train_df_mapped, n_users, n_items):
    graph_df = train_df_mapped.filter(pl.col('both_seen'))

    uids = graph_df['uid'].to_numpy()
    iids = graph_df['iid'].to_numpy()
    n = n_users + n_items

    rows = np.concatenate([uids, iids + n_users])
    cols = np.concatenate([iids + n_users, uids])
    data = np.ones(len(rows), dtype=np.float32)

    A = sp.csr_matrix((data, (rows, cols)), shape=(n, n))

    deg = np.array(A.sum(axis=1)).flatten()
    d_inv = np.where(deg > 0, deg ** -0.5, 0.0)
    D_inv = sp.diags(d_inv)
    A_hat = D_inv @ A @ D_inv

    A_coo = A_hat.tocoo()
    indices = torch.from_numpy(np.vstack([A_coo.row, A_coo.col]).astype(np.int64))
    values  = torch.from_numpy(A_coo.data.astype(np.float32))
    sparse_A = torch.sparse_coo_tensor(indices, values, (n, n)).coalesce()
    return sparse_A

print('Building normalized adjacency...')
t0 = time.time()
norm_adj = build_norm_adj(train_mapped, N_USERS, N_ITEMS).to(DEVICE)
print(f'Done in {time.time() - t0:.1f}s | nnz={norm_adj._nnz():,}')

Building normalized adjacency...
Done in 7.4s | nnz=29,858,446


In [36]:
# CELL 8 - LightGCN Model
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers, global_mean):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.global_mean = global_mean

        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def propagate(self, norm_adj):
        E0 = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        all_embs = [E0]
        E = E0

        for _ in range(self.n_layers):
            E = torch.sparse.mm(norm_adj, E)
            all_embs.append(E)

        E_final = torch.stack(all_embs, dim=1).mean(dim=1)
        user_final = E_final[:self.n_users]
        item_final = E_final[self.n_users:]
        return user_final, item_final

    def predict_with_cached_embeddings(self, uid, iid, user_final, item_final):
        e_u = user_final[uid]
        e_i = item_final[iid]
        dot = (e_u * e_i).sum(dim=1)
        b_u = self.user_bias(uid).squeeze(1)
        b_i = self.item_bias(iid).squeeze(1)
        return self.global_mean + b_u + b_i + dot

    def forward(self, uid, iid, norm_adj):
        user_final, item_final = self.propagate(norm_adj)
        return self.predict_with_cached_embeddings(uid, iid, user_final, item_final)

In [51]:
# CELL 9 - Dataset and DataLoaders

NUM_WORKERS = 4
PIN_MEMORY = (DEVICE.type == 'cuda')

class RatingDataset(Dataset):
    def __init__(self, df_mapped):
        df_seen = df_mapped.filter(pl.col('both_seen'))
        self.uids = torch.tensor(df_seen['uid'].to_numpy(), dtype=torch.long)
        self.iids = torch.tensor(df_seen['iid'].to_numpy(), dtype=torch.long)
        self.ratings = torch.tensor(df_seen['Rating'].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.uids[idx], self.iids[idx], self.ratings[idx]

train_loader = DataLoader(
    RatingDataset(train_mapped),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader_seen = DataLoader(
    RatingDataset(val_mapped),
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0)
)

print(f"NUM_WORKERS = {NUM_WORKERS}")
print(f"Train batches per epoch: {len(train_loader):,}")

NUM_WORKERS = 4
Train batches per epoch: 1,823


In [52]:
# CELL 10 - Fallback Prediction Helpers
user_mean_map = dict(zip(user_mean_df['user_id'].to_list(), user_mean_df['user_mean'].to_list()))
item_mean_map = dict(zip(item_mean_df['item_id'].to_list(), item_mean_df['item_mean'].to_list()))

def fallback_predict_row(user_id, item_id):
    item_mean = item_mean_map.get(item_id)
    user_mean = user_mean_map.get(user_id)

    if item_mean is not None:
        return float(item_mean)
    if user_mean is not None:
        return float(user_mean)
    return float(GLOBAL_MEAN)

def add_fallback_predictions(df):
    preds = [fallback_predict_row(u, i) for u, i in zip(df['user_id'].to_list(), df['item_id'].to_list())]
    return df.with_columns(pl.Series('fallback_pred', preds))

In [53]:
# CELL 11 - Shared Evaluation Function (match LightGBM style)
def calculate_metrics_polars(df, pred_col='predicted', k=10, threshold=7.0):
    sorted_df = df.sort(['user_id', pred_col], descending=[False, True])

    ranked_df = sorted_df.with_columns(
        pl.col(pred_col).rank(method='ordinal', descending=True).over('user_id').alias('rank')
    )

    top_k = ranked_df.filter(pl.col('rank') <= k)

    user_metrics = ranked_df.group_by('user_id').agg([
        pl.col('Rating').filter(pl.col('Rating') >= threshold).count().alias('n_rel'),
        pl.col('Rating')
          .filter((pl.col('rank') <= k) & (pl.col('Rating') >= threshold))
          .count()
          .alias('n_rel_and_rec_k')
    ])

    user_metrics = user_metrics.with_columns([
        (pl.col('n_rel_and_rec_k') / k).alias('precision'),
        (
            pl.when(pl.col('n_rel') > 0)
              .then(pl.col('n_rel_and_rec_k') / pl.col('n_rel'))
              .otherwise(0.0)
        ).alias('recall')
    ])

    top_k_dcg = top_k.with_columns(
        (pl.col('Rating') / np.log2(pl.col('rank') + 1)).alias('dcg_item')
    ).group_by('user_id').agg(
        pl.col('dcg_item').sum().alias('dcg')
    )

    ideal_top_k = ranked_df.sort(['user_id', 'Rating'], descending=[False, True]).with_columns(
        pl.col('Rating').rank(method='ordinal', descending=True).over('user_id').alias('ideal_rank')
    ).filter(pl.col('ideal_rank') <= k)

    top_k_idcg = ideal_top_k.with_columns(
        (pl.col('Rating') / np.log2(pl.col('ideal_rank') + 1)).alias('idcg_item')
    ).group_by('user_id').agg(
        pl.col('idcg_item').sum().alias('idcg')
    )

    ndcg_df = top_k_dcg.join(top_k_idcg, on='user_id', how='inner').with_columns(
        (
            pl.when(pl.col('idcg') > 0)
              .then(pl.col('dcg') / pl.col('idcg'))
              .otherwise(0.0)
        ).alias('ndcg')
    )

    rmse = float(np.sqrt(((df['Rating'] - df[pred_col]) ** 2).mean()))

    return {
        'RMSE': rmse,
        'Precision@10': float(user_metrics['precision'].mean()),
        'Recall@10': float(user_metrics['recall'].mean()),
        'NDCG@10': float(ndcg_df['ndcg'].mean())
    }

In [54]:
# CELL 12 - Validation Prediction Function for Full Split
def predict_full_split(model, df_mapped, norm_adj, batch_size=16384):
    model.eval()

    df_base = add_fallback_predictions(df_mapped)

    both_seen_df = df_base.filter(pl.col('both_seen'))
    unseen_df = df_base.filter(~pl.col('both_seen'))

    pred_seen = []

    with torch.no_grad():
        user_final, item_final = model.propagate(norm_adj)
        user_bias = model.user_bias.weight.squeeze(1)
        item_bias = model.item_bias.weight.squeeze(1)

        uid_arr = both_seen_df['uid'].to_numpy()
        iid_arr = both_seen_df['iid'].to_numpy()

        for start in range(0, len(both_seen_df), batch_size):
            end = start + batch_size
            uid = torch.tensor(uid_arr[start:end], dtype=torch.long, device=DEVICE)
            iid = torch.tensor(iid_arr[start:end], dtype=torch.long, device=DEVICE)

            preds = (
                model.global_mean
                + user_bias[uid]
                + item_bias[iid]
                + (user_final[uid] * item_final[iid]).sum(dim=1)
            ).clamp(1, 10)

            pred_seen.extend(preds.detach().cpu().numpy().tolist())

    both_seen_df = both_seen_df.with_columns(pl.Series('predicted', pred_seen))
    unseen_df = unseen_df.with_columns(pl.col('fallback_pred').clip(1.0, 10.0).alias('predicted'))

    result = pl.concat([
        both_seen_df.select(['user_id', 'item_id', 'Rating', 'predicted']),
        unseen_df.select(['user_id', 'item_id', 'Rating', 'predicted'])
    ])

    return result

In [55]:
# CELL 13 - Train One Configuration
VAL_SAMPLE_SIZE = 100_000

val_sample_for_search = (
    val_mapped.sample(n=min(VAL_SAMPLE_SIZE, len(val_mapped)), seed=42)
    if len(val_mapped) > VAL_SAMPLE_SIZE else val_mapped
)

def predict_full_split(model, df_mapped, norm_adj, batch_size=16384):
    model.eval()

    df_base = add_fallback_predictions(df_mapped)

    both_seen_df = df_base.filter(pl.col('both_seen'))
    unseen_df = df_base.filter(~pl.col('both_seen'))

    pred_seen = []

    with torch.no_grad():
        user_final, item_final = model.propagate(norm_adj)
        uid_arr = both_seen_df['uid'].to_numpy()
        iid_arr = both_seen_df['iid'].to_numpy()

        for start in range(0, len(both_seen_df), batch_size):
            end = start + batch_size
            uid = torch.tensor(uid_arr[start:end], dtype=torch.long, device=DEVICE)
            iid = torch.tensor(iid_arr[start:end], dtype=torch.long, device=DEVICE)

            preds = model.predict_with_cached_embeddings(uid, iid, user_final, item_final).clamp(1, 10)
            pred_seen.extend(preds.detach().cpu().numpy().tolist())

    both_seen_df = both_seen_df.with_columns(pl.Series('predicted', pred_seen))
    unseen_df = unseen_df.with_columns(
        pl.col('fallback_pred').clip(1.0, 10.0).alias('predicted')
    )

    result = pl.concat([
        both_seen_df.select(['user_id', 'item_id', 'Rating', 'predicted']),
        unseen_df.select(['user_id', 'item_id', 'Rating', 'predicted'])
    ])

    return result

def train_one_config(emb_dim, n_layers, lr):
    model = LightGCN(
        n_users=N_USERS,
        n_items=N_ITEMS,
        emb_dim=emb_dim,
        n_layers=n_layers,
        global_mean=GLOBAL_MEAN
    ).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()

    best_val_rmse = float('inf')
    best_state = None
    patience_ctr = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        n_batches = 0
        t0 = time.time()

        for uid, iid, rating in train_loader:
            uid = uid.to(DEVICE, non_blocking=True)
            iid = iid.to(DEVICE, non_blocking=True)
            rating = rating.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            preds = model(uid, iid, norm_adj).clamp(1, 10)
            loss = criterion(preds, rating)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        val_pred_df = predict_full_split(model, val_sample_for_search, norm_adj)
        val_metrics = calculate_metrics_polars(
            val_pred_df,
            pred_col='predicted',
            k=K_METRICS,
            threshold=THRESHOLD
        )
        val_rmse = val_metrics['RMSE']

        print(
            f'emb_dim={emb_dim}, layers={n_layers}, lr={lr:.0e} | '
            f'Epoch {epoch:02d} | TrainLoss={total_loss / n_batches:.4f} | '
            f'Val RMSE={val_rmse:.4f} | {time.time() - t0:.0f}s'
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

        gc.collect()
        torch.cuda.empty_cache()

    return best_val_rmse, best_state

In [48]:
# CELL 14 - Hyperparameter Search
print('Starting validation search...')

search_results = []
best_overall_rmse = float('inf')
best_overall_state = None
best_config = None

for emb_dim, n_layers, lr in itertools.product(
    EMB_DIM_OPTIONS, N_LAYER_OPTIONS, LR_OPTIONS
):
    print('\n' + '=' * 80)
    print(f'Testing config: emb_dim={emb_dim}, n_layers={n_layers}, lr={lr}')
    print('=' * 80)

    val_rmse, state = train_one_config(emb_dim, n_layers, lr)

    search_results.append({
        'emb_dim': emb_dim,
        'n_layers': n_layers,
        'lr': lr,
        'weight_decay': WEIGHT_DECAY,
        'best_val_rmse': val_rmse
    })

    if val_rmse < best_overall_rmse:
        best_overall_rmse = val_rmse
        best_overall_state = state
        best_config = {
            'emb_dim': emb_dim,
            'n_layers': n_layers,
            'lr': lr,
            'weight_decay': WEIGHT_DECAY
        }

search_results_df = pl.DataFrame(search_results).sort('best_val_rmse')

print('\nValidation search complete.')
print(search_results_df)
print('\nBest config:', best_config)
print(f'Best validation RMSE: {best_overall_rmse:.4f}')

Starting validation search...

Testing config: emb_dim=32, n_layers=2, lr=0.001
emb_dim=32, layers=2, lr=1e-03 | Epoch 01 | TrainLoss=1.9171 | Val RMSE=1.3023 | 171s
emb_dim=32, layers=2, lr=1e-03 | Epoch 02 | TrainLoss=1.6342 | Val RMSE=1.2524 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 03 | TrainLoss=1.5491 | Val RMSE=1.2317 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 04 | TrainLoss=1.5100 | Val RMSE=1.2209 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 05 | TrainLoss=1.4884 | Val RMSE=1.2147 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 06 | TrainLoss=1.4756 | Val RMSE=1.2110 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 07 | TrainLoss=1.4680 | Val RMSE=1.2089 | 169s
emb_dim=32, layers=2, lr=1e-03 | Epoch 08 | TrainLoss=1.4634 | Val RMSE=1.2075 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 09 | TrainLoss=1.4605 | Val RMSE=1.2066 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 10 | TrainLoss=1.4586 | Val RMSE=1.2061 | 170s
emb_dim=32, layers=2, lr=1e-03 | Epoch 11 | TrainLoss=1.4575

KeyboardInterrupt: 

In [56]:
best_config = {
    'emb_dim': 32,
    'n_layers': 2,
    'lr': 3e-3,
    'weight_decay': 1e-5
}

best_overall_rmse, best_overall_state = train_one_config(
    best_config['emb_dim'],
    best_config['n_layers'],
    best_config['lr']
)

emb_dim=32, layers=2, lr=3e-03 | Epoch 01 | TrainLoss=1.7126 | Val RMSE=1.2336 | 170s
emb_dim=32, layers=2, lr=3e-03 | Epoch 02 | TrainLoss=1.5004 | Val RMSE=1.2118 | 169s
emb_dim=32, layers=2, lr=3e-03 | Epoch 03 | TrainLoss=1.4716 | Val RMSE=1.2071 | 170s
emb_dim=32, layers=2, lr=3e-03 | Epoch 04 | TrainLoss=1.4647 | Val RMSE=1.2058 | 169s
emb_dim=32, layers=2, lr=3e-03 | Epoch 05 | TrainLoss=1.4626 | Val RMSE=1.2053 | 170s
emb_dim=32, layers=2, lr=3e-03 | Epoch 06 | TrainLoss=1.4619 | Val RMSE=1.2053 | 169s
emb_dim=32, layers=2, lr=3e-03 | Epoch 07 | TrainLoss=1.4618 | Val RMSE=1.2050 | 170s
emb_dim=32, layers=2, lr=3e-03 | Epoch 08 | TrainLoss=1.4616 | Val RMSE=1.2050 | 169s
emb_dim=32, layers=2, lr=3e-03 | Epoch 09 | TrainLoss=1.4615 | Val RMSE=1.2050 | 169s
emb_dim=32, layers=2, lr=3e-03 | Epoch 10 | TrainLoss=1.4616 | Val RMSE=1.2048 | 170s
emb_dim=32, layers=2, lr=3e-03 | Epoch 11 | TrainLoss=1.4614 | Val RMSE=1.2051 | 170s
emb_dim=32, layers=2, lr=3e-03 | Epoch 12 | TrainLoss=

In [57]:
# CELL 15 - Load Best Model and Evaluate on Test with Shared Metrics
best_model = LightGCN(
    n_users=N_USERS,
    n_items=N_ITEMS,
    emb_dim=best_config['emb_dim'],
    n_layers=best_config['n_layers'],
    global_mean=GLOBAL_MEAN
).to(DEVICE)

best_model.load_state_dict(best_overall_state)

print('Generating full test predictions...')
test_pred_df = predict_full_split(best_model, test_mapped, norm_adj)

print('Calculating final shared metrics...')
final_metrics = calculate_metrics_polars(
    test_pred_df,
    pred_col='predicted',
    k=K_METRICS,
    threshold=THRESHOLD
)

print('\n' + '=' * 55)
print('FINAL LIGHTGCN RESULTS (SHARED TEST EVALUATION)')
print('=' * 55)
for metric, value in final_metrics.items():
    print(f'{metric:12}: {value:.4f}')
print('=' * 55)

Generating full test predictions...
Calculating final shared metrics...

FINAL LIGHTGCN RESULTS (SHARED TEST EVALUATION)
RMSE        : 1.2755
Precision@10: 0.2673
Recall@10   : 0.9060
NDCG@10     : 0.9848


In [59]:
# CELL 16 - Save Best Model and Search Results
torch.save({
    'model_state': best_overall_state,
    'user2idx': user2idx,
    'item2idx': item2idx,
    'best_config': best_config,
    'global_mean': GLOBAL_MEAN,
    'n_users': N_USERS,
    'n_items': N_ITEMS
}, MODEL_PATH)

SEARCH_RESULTS_PATH = f'{DRIVE_PATH}/processed_data/lightgcn_search_results.parquet'
#search_results_df.write_parquet(SEARCH_RESULTS_PATH)

print(f'Model saved to: {MODEL_PATH}')
print(f'Search results saved to: {SEARCH_RESULTS_PATH}')

Model saved to: /content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/processed_data/lightgcn_best_model.pt
Search results saved to: /content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/processed_data/lightgcn_search_results.parquet


In [60]:
# CELL 17 - Comparison Table
baselines = [
    ('Popularity-Based', 'Baseline', 1.3781, 0.2672, 0.9060, 0.9848),
    ('SVD', 'Baseline', 1.2960, 0.2297, 0.7563, 0.9834),
    (
        'LightGCN',
        'Darshan',
        round(final_metrics['RMSE'], 4),
        round(final_metrics['Precision@10'], 4),
        round(final_metrics['Recall@10'], 4),
        round(final_metrics['NDCG@10'], 4),
    )
]

print('\n{:<18} {:<10} {:>8} {:>14} {:>11} {:>10}'.format(
    'Model', 'Author', 'RMSE', 'Precision@10', 'Recall@10', 'NDCG@10'
))
print('-' * 80)

for row in baselines:
    print('{:<18} {:<10} {:>8.4f} {:>14.4f} {:>11.4f} {:>10.4f}'.format(*row))


Model              Author         RMSE   Precision@10   Recall@10    NDCG@10
--------------------------------------------------------------------------------
Popularity-Based   Baseline     1.3781         0.2672      0.9060     0.9848
SVD                Baseline     1.2960         0.2297      0.7563     0.9834
LightGCN           Darshan      1.2755         0.2673      0.9060     0.9848
